# AudioSR — MP3 音声アップスケール＆ノイズ補完 🎧

**haoheliu/versatile_audio_super_resolution** ⭐1884

どんな音声でも **48kHz 高音質** にアップスケール。拡散モデルで失われた高域を復元。

- MP3 の劣化した高域を補完
- ノイズ除去 + 帯域拡張
- 音楽・音声・効果音なんでもOK

---
### 🔄 処理の流れ
1. ローカルから音声ファイルをアップロード
2. MP3 の場合は前処理（ローパスフィルター）を自動適用
3. AudioSR で 48kHz アップスケール
4. 元の音声と比較して聴ける＆ダウンロード

## ⚙ セットアップ

ランタイム → **T4 GPU** を選択推奨（CPU でも動くけど10倍遅い）

In [ ]:
# 依存関係インストール（初回のみ）
!pip install -q audiosr==0.0.7
!pip install -q gradio==4.44.1

import os, sys, warnings
warnings.filterwarnings('ignore')
print(f'✅ インストール完了 | Python {sys.version}')

## 📁 音声ファイルをアップロード

対応形式: **MP3 / WAV / FLAC / OGG / M4A**

In [ ]:
from google.colab import files
import librosa, soundfile as sf, numpy as np, IPython.display as ipd

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('ファイルがアップロードされませんでした')

input_path = list(uploaded.keys())[0]
input_name, input_ext = os.path.splitext(input_path)

audio, sr = librosa.load(input_path, sr=None, mono=False)
duration = len(audio) / sr if audio.ndim == 1 else len(audio[0]) / sr
n_ch = 1 if audio.ndim == 1 else audio.shape[0]

print(f'📂 {input_path}')
print(f'   • {sr}Hz / {n_ch}ch / {duration:.1f}秒 / {input_ext.upper()}')

wav_path = '/content/input_audio.wav'
sf.write(wav_path, audio, sr)
print(f'✅ WAV変換: {wav_path}')

## 👂 元の音声

In [ ]:
print(f'🎧 元の音声（{sr}Hz / {duration:.1f}秒）')
ipd.display(ipd.Audio(wav_path))

## 🛠 MP3 前処理（ローパスフィルター）

**なぜ必要？** AudioSR の学習データはローパスシミュレーション由来。MP3 の特殊なカットオフパターンは認識できません。
→ 事前にローパスフィルターをかけて「なだらかな減衰」に変換します。

In [ ]:
from scipy.signal import butter, sosfilt
import matplotlib.pyplot as plt

def lowpass_filter(audio, sr, cutoff_hz=8000, order=6):
    sos = butter(order, cutoff_hz, btype='low', fs=sr, output='sos')
    if audio.ndim == 1:
        return sosfilt(sos, audio)
    return np.array([sosfilt(sos, audio[ch]) for ch in range(audio.shape[0])])

def plot_spec(audio_data, sr, title):
    plt.figure(figsize=(12, 3))
    if audio_data.ndim > 1:
        audio_data = np.mean(audio_data, axis=0)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio_data)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.ylim(0, 24000)
    plt.tight_layout(); plt.show()

is_mp3 = input_ext.lower() in ['.mp3', '.m4a']

if is_mp3:
    print('🎯 MP3 → ローパスフィルター (8kHz) 適用')
    audio_filtered = lowpass_filter(audio, sr)
    input_for_audiosr = '/content/input_preprocessed.wav'
    sf.write(input_for_audiosr, audio_filtered, sr)
    plot_spec(audio, sr, f'元の MP3 ({sr/1000:.0f}kHz)')
    plot_spec(audio_filtered, sr, f'ローパス後 (8kHz カット)')
else:
    input_for_audiosr = wav_path
    plot_spec(audio, sr, f'入力 ({sr/1000:.0f}kHz)')

## 🚀 AudioSR 実行

In [ ]:
# === 設定 ===
MODEL_NAME = 'basic'       # 'basic' か 'speech'
DDIM_STEPS = 50            # 25=高速, 50=標準, 100=高品質
GUIDANCE_SCALE = 3.5       # 2.5〜5.0 推奨
DEVICE = 'cuda'
OUTPUT_DIR = '/content/output'
# ===========

os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
if not torch.cuda.is_available() and DEVICE == 'cuda':
    print('⚠ GPU 未検出 → CPU（10〜30倍遅）')
    DEVICE = 'cpu'
else:
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'📋 {MODEL_NAME} / DDIM={DDIM_STEPS} / GS={GUIDANCE_SCALE} / {DEVICE}')

In [ ]:
print(f'⏳ AudioSR 処理中...（推定 {duration*0.3:.0f}〜{duration*1:.0f}秒）')
!audiosr -i "{input_for_audiosr}" -s "{OUTPUT_DIR}" --model_name {MODEL_NAME} --ddim_steps {DDIM_STEPS} -gs {GUIDANCE_SCALE} -d {DEVICE} --suffix "_audiosr"
print('✅ AudioSR 完了！')

## 🎉 結果比較

スペクトログラムで高域復元を確認 👇

In [ ]:
output_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.wav')]
if not output_files:
    raise FileNotFoundError('出力ファイルなし')

output_path = os.path.join(OUTPUT_DIR, output_files[0])
audio_sr, sr_sr = librosa.load(output_path, sr=None, mono=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
src = audio if not is_mp3 else audio
if src.ndim > 1: src = np.mean(src, axis=0)
D0 = librosa.amplitude_to_db(np.abs(librosa.stft(src)), ref=np.max)
librosa.display.specshow(D0, sr=sr, ax=axes[0], x_axis='time', y_axis='hz')
axes[0].set_title(f'入力（{sr/1000:.0f}kHz）'); axes[0].set_ylim(0, 24000)
m = audio_sr if audio_sr.ndim == 1 else np.mean(audio_sr, axis=0)
D1 = librosa.amplitude_to_db(np.abs(librosa.stft(m)), ref=np.max)
librosa.display.specshow(D1, sr=sr_sr, ax=axes[1], x_axis='time', y_axis='hz')
axes[1].set_title(f'AudioSR（{sr_sr/1000:.0f}kHz）'); axes[1].set_ylim(0, 24000)
plt.colorbar(axes[1].collections[0], ax=axes, format='%+2.0f dB')
plt.tight_layout(); plt.show()

print('🎧 元の音声:'); ipd.display(ipd.Audio(wav_path))
print(f'🎧 AudioSR（{sr_sr/1000:.0f}kHz）:'); ipd.display(ipd.Audio(output_path))

## 💾 ダウンロード

In [ ]:
from google.colab import files
print('🔄 MP3変換...')
dl = '/content/audiosr_output.mp3'
!ffmpeg -i "{output_path}" -codec:a libmp3lame -b:a 320k "{dl}" -y 2>/dev/null
files.download(dl)

In [ ]:
# WAV版（ロスレス）も欲しい場合
from google.colab import files
files.download(output_path)

---
## 🎨 Gradio Web UI モード

コード不要の Web UI。↓実行するだけでブラウザから使える。
`share=True` により Colab 外からもアクセス可能な一時URLが発行されます。

In [ ]:
import gradio as gr
import subprocess
import librosa, soundfile as sf, numpy as np
from scipy.signal import butter, sosfilt

def lowpass(audio, sr, cutoff=8000, order=6):
    sos = butter(order, cutoff, btype='low', fs=sr, output='sos')
    return sosfilt(sos, audio) if audio.ndim == 1 else np.array([sosfilt(sos, audio[c]) for c in range(audio.shape[0])])

def process(path, model, steps, gs):
    os.makedirs('/tmp/ao', exist_ok=True)
    a, r = librosa.load(path, sr=None, mono=False)
    if os.path.splitext(path)[1].lower() in ('.mp3','.m4a'): a = lowpass(a, r)
    sf.write('/tmp/ao/in.wav', a, r)
    p = subprocess.run(['audiosr','-i','/tmp/ao/in.wav','-s','/tmp/ao','--model_name',model,'--ddim_steps',str(steps),'-gs',str(gs),'-d','cuda'], capture_output=True, text=True, timeout=600)
    if p.returncode != 0: return None, f'Error: {p.stderr[:200]}'
    o = [f for f in os.listdir('/tmp/ao') if f.endswith('.wav')]
    if not o: return None, '出力なし'
    subprocess.run(['ffmpeg','-i',os.path.join('/tmp/ao',o[0]),'-codec:a','libmp3lame','-b:a','320k','/tmp/ao/out.mp3','-y'], capture_output=True, timeout=60)
    return '/tmp/ao/out.mp3', f'✅ {r/1000:.0f}kHz→48kHz'

with gr.Blocks(title='AudioSR', theme=gr.themes.Soft()) as d:
    gr.Markdown('# 🎧 AudioSR アップスケーラー')
    with gr.Row():
        with gr.Column():
            i = gr.Audio(label='ファイル', type='filepath')
            with gr.Accordion('⚙', open=False):
                m = gr.Radio(['basic','speech'], value='basic', label='モデル')
                s = gr.Slider(15, 100, 50, 5, label='DDIM')
                g = gr.Slider(1, 10, 3.5, 0.5, label='GS')
            b = gr.Button('🚀 実行', variant='primary')
        with gr.Column():
            st = gr.Textbox(label='状態', interactive=False)
            o = gr.Audio(label='結果', type='filepath', interactive=False)
    b.click(fn=process, inputs=[i,m,s,g], outputs=[o,st])

print('🎯 起動中...')
d.launch(share=True, debug=False)

---
## 💡 Tips

| 設定 | 効果 |
|---|---|
| DDIM 50 + GS 3.5 | 標準 |
| DDIM 25 + GS 2.5 | 高速 |
| DDIM 100 + GS 5.0 | 最高品質 |
| model=speech | 音声特化 |
| model=basic | 汎用（音楽・効果音） |

**⚠ MP3 は自動ローパス処理** | 初回はモデルDL(〜2GB)あり

**[GitHub](https://github.com/haoheliu/versatile_audio_super_resolution) | [arXiv](https://arxiv.org/abs/2309.07314)**